# Q1 — Text Classification (IMDb)
**Models:** TF-IDF LR/SVM (CPU) → BiLSTM (GPU) → DistilBERT (GPU)

**Dataset:** Full 25K IMDb training set

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q1'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
# Clone repo
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run TF-IDF Models (CPU) — LR + SVM

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=true \
        models.tfidf_svm.enabled=true \
        models.bilstm.enabled=false \
        models.distilbert.enabled=false

In [ ]:
# Save TF-IDF results to Drive
import glob, shutil
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_tfidf')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'TF-IDF results saved to Drive: {dest}')

## 3. Run BiLSTM (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=true \
        models.distilbert.enabled=false \
        models.bilstm.max_epochs=10 \
        models.bilstm.batch_size=64

In [ ]:
# Save BiLSTM results to Drive
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_bilstm')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'BiLSTM results saved to Drive: {dest}')

## 4. Run DistilBERT (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=false \
        models.distilbert.enabled=true \
        models.distilbert.max_epochs=3 \
        models.distilbert.batch_size=16

In [ ]:
# Save DistilBERT results to Drive
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_distilbert')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'DistilBERT results saved to Drive: {dest}')

## 5. Results Summary

In [ ]:
import json

print('=== Q1 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q1/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:1500])